# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,Chinese wine tempts Italy's Illva Italy's Illv...,0
1,Labour chooses Manchester The Labour Party wil...,2
2,Iran budget seeks state sell-offs Iran's presi...,0
3,Roundabout continues nostalgia trip The new bi...,1
4,US charity anthem is re-released We Are The Wo...,1
...,...,...
1507,Game warnings 'must be clearer' Violent video ...,2
1508,Blair ready to call election Tony Blair seems ...,2
1509,Mourinho expects fight to finish Chelsea manag...,3
1510,India power shares jump on debut Shares in Ind...,0


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        # For multiclass, output_dim = number_of_classes
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        # If bidirectional=True, hidden vectors are doubled in size
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1) Embedding lookup
        embedded = self.embedding(input_ids)
        # 2) LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3) Extract final hidden state
        if self.lstm.bidirectional:
            # concatenate forward & backward final hidden states
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4) Dropout
        hidden = self.dropout(hidden)

        # 5) Fully connected layer -> logits of shape [batch_size, output_dim]
        output = self.fc(hidden)
        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = test_df['label'].nunique()
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass -> logits: [batch_size, num_classes]
        logits = model(input_ids)
        # CrossEntropyLoss expects [batch_size, num_classes] vs. [batch_size] labels
        loss = criterion(logits, labels)

        # Backprop and optimize
        loss.backward()
        optimizer.step()

        # Track loss
        losses.append(loss.item())

        # Convert logits -> predicted classes
        preds_cls = torch.argmax(logits, dim=1)

        # Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # Accumulate predictions and labels for metric calculations
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate average loss and overall accuracy
    avg_loss = sum(losses) / len(losses)
    accuracy = float(correct_predictions) / len(data_loader.dataset)

    # Calculate macro metrics
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Calculate per-class F1-scores
    # This will return a NumPy array of length = num_classes
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class

def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds_cls = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    avg_loss = sum(losses) / len(losses)
    accuracy = float(correct_predictions) / len(data_loader.dataset)

    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class


# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        
        val_acc, val_loss, val_prec, val_rec, val_f1_macro, val_f1_per_class = eval_model(
            model, val_loader, criterion, device
        )
        
        # Print macro stats
        print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, "
              f"Precision(macro): {train_prec:.4f}, Recall(macro): {train_rec:.4f}, "
              f"F1(macro): {train_f1_macro:.4f}")
        
        # Print per-class F1 for train
        print(f"F1 Per Class (Train): {train_f1_per_class}")
        
        print(f"Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, "
              f"Precision(macro): {val_prec:.4f}, Recall(macro): {val_rec:.4f}, "
              f"F1(macro): {val_f1_macro:.4f}")
        
        # Print per-class F1 for val
        print(f"F1 Per Class (Val):   {val_f1_per_class}")
        print("--------------------------------------------------")
    
    # Return the final metrics from the last epoch, if you like
    return (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class
    )


In [10]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=[
    'seed', 
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',
    'test_loss',  'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class = retval

    new_row = pd.DataFrame([[
        seed,
        train_loss, train_acc, train_prec, train_rec, train_f1_macro, train_f1_per_class,
        val_loss,   val_acc,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,
        test_loss,  test_acc,  test_prec,  test_rec,  test_f1,  test_f1_per_class,
        max_memory_usage_train, max_vram_usage_train, total_time_train,
        max_memory_usage_test,  max_vram_usage_test,  total_time_test
    ]], columns=results.columns)

    results = pd.concat([results, new_row], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 1.5704, Accuracy: 0.2619, Precision(macro): 0.2739, Recall(macro): 0.2395, F1(macro): 0.2130
F1 Per Class (Train): [0.3403895  0.08284024 0.24935065 0.30649351 0.08571429]
Val Loss: 1.4423, Accuracy: 0.3456, Precision(macro): 0.4791, Recall(macro): 0.3564, F1(macro): 0.2988
F1 Per Class (Val):   [0.14583333 0.30357143 0.5        0.28571429 0.25882353]
--------------------------------------------------
Epoch 2/5
Train Loss: 1.1046, Accuracy: 0.6019, Precision(macro): 0.5972, Recall(macro): 0.5997, F1(macro): 0.5970
F1 Per Class (Train): [0.56692913 0.48091603 0.62676056 0.66941015 0.64084507]
Val Loss: 1.0055, Accuracy: 0.6385, Precision(macro): 0.6456, Recall(macro): 0.6228, F1(macro): 0.6243
F1 Per Class (Val):   [0.64864865 0.49180328 0.67164179 0.7254902  0.5840708 ]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.5498, Accuracy: 0.8049, Precision(macro): 0.8015, Recall(macro): 0.7988, F1(macro): 0.7997
F1 Per Class (Train): [0.817142

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_21732\2112283703.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row], ignore_index=True)
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 1.5831, Accuracy: 0.2738, Precision(macro): 0.2884, Recall(macro): 0.2502, F1(macro): 0.2194
F1 Per Class (Train): [0.30853392 0.05245902 0.27817746 0.3593603  0.09846154]
Val Loss: 1.4658, Accuracy: 0.4116, Precision(macro): 0.4935, Recall(macro): 0.4013, F1(macro): 0.3660
F1 Per Class (Val):   [0.38216561 0.11267606 0.51485149 0.46296296 0.35714286]
--------------------------------------------------
Epoch 2/5
Train Loss: 1.0731, Accuracy: 0.5926, Precision(macro): 0.5909, Recall(macro): 0.5888, F1(macro): 0.5869
F1 Per Class (Train): [0.53978159 0.51345756 0.65505226 0.65374677 0.57246377]
Val Loss: 0.9172, Accuracy: 0.6913, Precision(macro): 0.7081, Recall(macro): 0.6787, F1(macro): 0.6751
F1 Per Class (Val):   [0.69364162 0.57364341 0.73417722 0.78571429 0.58823529]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.4857, Accuracy: 0.8333, Precision(macro): 0.8313, Recall(macro): 0.8298, F1(macro): 0.8304
F1 Per Class (Train): [0.818443

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 1.5674, Accuracy: 0.2784, Precision(macro): 0.2712, Recall(macro): 0.2529, F1(macro): 0.2124
F1 Per Class (Train): [0.35266084 0.03378378 0.30117647 0.33468286 0.03960396]
Val Loss: 1.4912, Accuracy: 0.3061, Precision(macro): 0.4945, Recall(macro): 0.2761, F1(macro): 0.2038
F1 Per Class (Val):   [0.20512821 0.         0.36956522 0.41545894 0.02898551]
--------------------------------------------------
Epoch 2/5
Train Loss: 1.2509, Accuracy: 0.4967, Precision(macro): 0.4806, Recall(macro): 0.4800, F1(macro): 0.4611
F1 Per Class (Train): [0.49479167 0.16071429 0.60810811 0.57247259 0.46942801]
Val Loss: 1.1755, Accuracy: 0.5488, Precision(macro): 0.5499, Recall(macro): 0.5431, F1(macro): 0.5299
F1 Per Class (Val):   [0.52238806 0.33898305 0.65868263 0.63681592 0.49275362]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.7063, Accuracy: 0.7440, Precision(macro): 0.7371, Recall(macro): 0.7385, F1(macro): 0.7367
F1 Per Class (Train): [0.717201

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_multiclass2.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,train_f1_per_class,val_loss,val_acc,val_prec,...,test_prec,test_rec,test_f1,test_f1_per_class,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.101022,0.972222,0.971958,0.971225,0.971568,"[0.9814020028612304, 0.9576923076923077, 0.970...",1.008885,0.730871,0.745631,...,0.790456,0.760421,0.766861,"[0.7770700636942676, 0.7254901960784313, 0.779...",1194.101562,231.644531,15.764845,1193.570312,194.171875,1.005575
1,3,0.075108,0.978836,0.979024,0.979245,0.979118,"[0.9710144927536232, 0.9828571428571429, 0.982...",0.758278,0.765172,0.766847,...,0.780859,0.775541,0.776644,"[0.7581699346405228, 0.7090909090909091, 0.789...",1195.414062,232.115234,17.055370,1194.769531,194.254883,1.018948
2,5,0.137245,0.964947,0.964879,0.963955,0.964389,"[0.9583931133428981, 0.9441233140655106, 0.971...",0.942221,0.738786,0.755539,...,0.763387,0.736496,0.738202,"[0.7398843930635838, 0.7079646017699115, 0.666...",1211.070312,231.644531,17.001567,1209.707031,194.171875,0.995034


In [12]:
torch.save(model.state_dict(), 'results/lstm_multiclass2.pth')